In [39]:
!wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/rag_helper.py
!wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/ingest.py

--2026-07-22 09:39:26--  https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/rag_helper.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.108.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2134 (2.1K) [text/plain]
Saving to: ‘rag_helper.py.2’

rag_helper.py.2     100%[===================>]   2.08K  --.-KB/s    in 0s      

2026-07-22 09:39:26 (27.8 MB/s) - ‘rag_helper.py.2’ saved [2134/2134]

--2026-07-22 09:39:26--  https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/ingest.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.110.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 20

In [40]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [41]:
openai_client

In [42]:
from rag_helper import RAGBase
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

Create an assistant

In [43]:
instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
    instructions=instructions,
)

testing the assistant

In [44]:
answer = assistant.rag("How do I run Ollama locally?")
print(answer)

To run Ollama locally:

1. Install Ollama from https://ollama.com/download for your operating system.
   - macOS: install the `.pkg`
   - Windows: install the `.msi`
   - Linux: run
     ```bash
     curl -fsSL https://ollama.com/install.sh | sh
     ```

2. Open a terminal and run:
   ```bash
   ollama run llama3
   ```

This will download the LLaMA 3 model, start it locally, and open a chat-like interface.

To test that the local server is running, you can also run:
```bash
curl http://localhost:11434
```

If you get a connection refused error while doing the homework, restart the Ollama server with:
```bash
!nohup ollama serve > nohup.out 2>&1 &
```


In [45]:
answer = assistant.rag("How do I run Olama locally?")
print(answer)

I’m not sure what “Olama” refers to in the FAQ context, so I can’t give specific instructions for running it locally.

The closest relevant guidance in the FAQ is that you **can run the course locally instead of Codespaces** if you’re comfortable setting up **Python, `uv`, Jupyter, Docker, and any other tools needed for the module**. If you do run locally, you should **document your setup and keep it reproducible**.

If you meant a specific tool from the course, tell me which one and I’ll answer based on the FAQ context.


## Function Calling
In the previous lesson we built a RAG pipeline with `RAGBase.rag()` and saw it fail on the "Olama" typo. The search returned nothing useful, and the LLM had no way to recover.

The flow that broke:

```mermaid
flowchart TD
    U([User: How do I run Olama?])
    S[search - Olama - no useful results]
    A([LLM: I don't have information about Olama.])

    U --> S --> A

```

The pipeline is fixed: search, build prompt, LLM.

In [46]:
def rag(self, query):
    search_results = self.search(query)
    prompt = self.build_prompt(query, search_results)
    answer = self.llm(prompt)
    return answer

The LLM is a passenger here, not a driver. It never sees the bad search results, so it can't try again with a corrected query.

### The agent alternative
An agent puts the LLM in charge.

Instead of running search ourselves, we give the LLM a search tool. It decides when to call it and what to search for.

The same typo question now goes like this:

```mermaid
flowchart TD
    U([User: How do I run Olama?])
    L1[LLM: I'll search for 'Olama']
    S1[search - Olama - no useful results]
    L2[LLM: Hmm, no results. Maybe a typo for 'Ollama'?]
    S2[search - Ollama - found results!]
    A([LLM: Here's how to run Ollama locally...])

    U --> L1 --> S1 --> L2 --> S2 --> A
```

The LLM searched, saw the results were bad, and decided to try again with a different query. It made that decision on its own. We didn't write any code to handle typos.

The difference is about who makes the decisions:

- With RAG, the developer decides. We fix the steps up front, so search always runs once with the exact user query.
- With an agent, the LLM decides. It chooses which actions to take and when to stop.
The mechanism that makes this possible is function calling, and that's what the rest of this lesson is about.

### Asking without tools
First, let's see what the LLM does without any tools. We ask it a course-specific question and look at the answer.

In [47]:
messages = [
    {"role": "user", "content": "I just discovered the course. Can I join it?"}
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
)

response.output_text

'Maybe — it depends on the course’s enrollment rules and whether registration is still open.\n\nIf you want, I can help you figure it out quickly. Just send me:\n- the course name or link\n- the school/platform\n- whether it’s live, self-paced, or part of a program\n\nIf you’re asking what to say to the instructor, you could use:\n\n> Hi, I just discovered this course and I’m very interested in joining. Is it still possible to enroll? If so, could you please let me know the steps?\n\nIf you want, I can also help you write a more polished version.'

### Defining the tool
First we define a top-level search function that queries the index directly. The model will reference it by this name. We keep the Python function and the tool name aligned so the dispatch is easier later.

In [48]:
def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

Next we tell the model about this function. The model doesn't see our Python code, only a schema describing what the function does and what arguments it takes. LLMs are language agnostic. At the end we're just making an HTTP call, so we describe the tool in JSON rather than in Python. The same schema would work from TypeScript or Java.

In [49]:
search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

### Sending the question with the tool

In [50]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

response.output

[ResponseFunctionToolCall(arguments='{"query":"join course discovered late can I join enroll late start after course has started"}', call_id='call_7GGJobfi9ozL8OAvBQd0aJyc', name='search', type='function_call', id='fc_07cfbac8cabcb502006a608fd9da2481949f5481099c43cbac', namespace=None, status='completed')]

Look at the output. Instead of a message with the answer, the response contains a `function_call` entry. The model decided it needs to search the FAQ before answering. Rather than reply, it asked us to run the search function first.

Look at the arguments too. The model didn't pass our question verbatim. It judged the raw question wasn't the best query to search with. So it rewrote our enrollment question into search keywords like "enroll late join course".

In [51]:
call = response.output[0]

In [52]:
call.__dict__

{'arguments': '{"query":"join course discovered late can I join enroll late start after course has started"}',
 'call_id': 'call_7GGJobfi9ozL8OAvBQd0aJyc',
 'name': 'search',
 'type': 'function_call',
 'id': 'fc_07cfbac8cabcb502006a608fd9da2481949f5481099c43cbac',
 'namespace': None,
 'status': 'completed'}

In [53]:
call.arguments

'{"query":"join course discovered late can I join enroll late start after course has started"}'

In [54]:
call.name

'search'

In [55]:
call.type

'function_call'

In [56]:
import json

call = response.output[0]
args = json.loads(call.arguments)

results = search(**args)
result_json = json.dumps(results, indent=2)

In [57]:
messages.extend(response.output)

messages.append({
    "type": "function_call_output",
    "call_id": call.call_id,
    "output": result_json,
})

In [58]:
messages

[{'role': 'user', 'content': 'I just discovered the course. Can I join it?'},
 ResponseFunctionToolCall(arguments='{"query":"join course discovered late can I join enroll late start after course has started"}', call_id='call_7GGJobfi9ozL8OAvBQd0aJyc', name='search', type='function_call', id='fc_07cfbac8cabcb502006a608fd9da2481949f5481099c43cbac', namespace=None, status='completed'),
 {'type': 'function_call_output',
  'call_id': 'call_7GGJobfi9ozL8OAvBQd0aJyc',
  'output': '[\n  {\n    "id": "74eb249bbf",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "I just discovered the course. Can I still join?",\n    "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\\u2019re still accepting submissions."\n  },\n  {\n    "id": "04919992b3",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "How should I start the course and follow the weekly workfl

### Asking the model again
We call the API a second time with the expanded history:

In [60]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

print(response.output_text)

Yes — you can still join and start learning at any time.

If you want a certificate, though, you need to complete the capstone/project while the course is still accepting submissions.


This time the model has the original question, its own decision to call search, and the FAQ results. It can now produce a proper course-specific answer.

We have to send the whole history because LLMs are stateless between API calls. The memory is the list you send as input. If you send only the tool result, the model has no idea what's going on. So on this second call we replay everything we have so far. That means the question, the decision to call search, and the result we got back.

That's the full function-calling loop for a single turn. With plain RAG we made one call, and here we make two. Turning RAG agentic means more round-trips.

People call this pattern "agentic RAG", "tool use", or "function calling". The idea behind all of them is the same. The LLM decides which tools to call.

### Token usage and cost
We just made two API calls instead of one. Each call we send to the model costs money, so it's worth checking how much one tool-using turn actually costs.

The response has a `usage` field with the token counts:

In [61]:
usage = response.usage
usage.input_tokens, usage.output_tokens

(872, 41)

For each model the provider publishes a price per million input tokens and per million output tokens. Plug those numbers in to convert tokens to dollars.

In [62]:
def calculate_gpt54mini_price(input_tokens, output_tokens):
    INPUT_PRICE_PER_MILLION = 0.15
    OUTPUT_PRICE_PER_MILLION = 0.60

    input_cost = (input_tokens / 1_000_000) * INPUT_PRICE_PER_MILLION
    output_cost = (output_tokens / 1_000_000) * OUTPUT_PRICE_PER_MILLION
    total_cost = input_cost + output_cost

    return {
        "input_cost": input_cost,
        "output_cost": output_cost,
        "total_cost": total_cost,
    }

result = calculate_gpt54mini_price(652, 33)
print("Total cost: $", round(result["total_cost"], 8))

Total cost: $ 0.0001176
